# Vatican Ebr. 530 Luke and John: PDF to OSIS

Converts Nehemia Gordon's bilingual transcription, translation, and annotations of *Biblioteca Apostolica Vaticana ebr. 530*, part 1, fragment 11, folios 1r–2v. The PDF covers Luke 1:1–35 and John 1:1–13.

In [ ]:
from __future__ import annotations

from collections import OrderedDict
from pathlib import Path
import re
import sys
import unicodedata

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

import fitz  # PyMuPDF
from lxml import etree

PDF_PATH = Path('../data/00_source_files/Hebrew-Gospels-of-Luke-and-John-from-the-Vatican_Biblioteca Apostolica ebr. 530.pdf')
OUT_DIR = Path('../data/01_osis')
SCHEMA = Path('pdf2osis/schema/osisCore.2.1.1-project-subset.xsd')
OSIS_NS = 'http://www.bibletechnologies.net/2003/OSIS/namespace'
XSI_NS = 'http://www.w3.org/2001/XMLSchema-instance'

assert PDF_PATH.exists(), PDF_PATH
assert SCHEMA.exists(), SCHEMA
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Input: {PDF_PATH} ({len(fitz.open(PDF_PATH))} pages)')


In [ ]:
HEBREW_RE = re.compile(r'[\u0590-\u05ff]')
VERSE_ONLY_RE = re.compile(r'^\(?\s*(\d+)\s*\)?$')
ENGLISH_MARKER_RE = re.compile(r'^\(?\s*(\d+)\s*\)?\s*')

def normalise(text: str) -> str:
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', text)).strip()

def rtl_line(line: dict) -> str:
    return normalise(''.join(span['text'] for span in line.get('spans', []))[::-1])

def clean_hebrew(text: str) -> str:
    # Verse and superscript-note figures are encoded separately in OSIS.
    text = re.sub(r'[()]\s*\d+\s*[()]?|\d+\s*[()]|\d+', ' ', text)
    text = text.replace('(', ' ').replace(')', ' ')
    return normalise(text)

def append(record: dict, field: str, text: str) -> None:
    text = normalise(text)
    if text:
        record[field] = normalise(f"{record[field]} {text}")

def book_for_line(page_number: int, y: float) -> str | None:
    if 2 <= page_number <= 9:
        return 'Luke'
    if page_number == 10:
        if y < 240:
            return 'Luke'
        if y >= 410:
            return 'John'
        return None  # folio marker and John title/chapter heading
    if 11 <= page_number <= 12:
        return 'John'
    return None

def content_lines(page, side: str):
    """Return (y, text, verse_marker) from one printed column."""
    result = []
    for block in page.get_text('dict')['blocks']:
        if block.get('type') != 0:
            continue
        for line in block.get('lines', []):
            x0, y0, x1, y1 = line['bbox']
            if y0 < 60 or y1 > 610:
                continue
            spans = line.get('spans', [])
            spans = [s for s in spans if (s['bbox'][0] >= 306 if side == 'hebrew' else s['bbox'][2] <= 306)]
            if not spans:
                continue
            marker = None
            if side == 'hebrew':
                # Verse labels occupy their own right-edge span; embedded footnote
                # digits are not considered verse anchors.
                for span in spans:
                    match = VERSE_ONLY_RE.match(span['text'].strip())
                    if match and span['bbox'][0] >= 490:
                        marker = int(match.group(1))
                        break
                text = normalise(''.join(span['text'] for span in spans)[::-1])
                inline_marker = re.search(r'[()]\s*(\d+)\s*[()]', text)
                if inline_marker:
                    marker = int(inline_marker.group(1)[::-1])
            else:
                text = normalise(''.join(span['text'] for span in spans))
                match = ENGLISH_MARKER_RE.match(text)
                marker = int(match.group(1)) if match else None
            if text:
                result.append((y0, text, marker))
    return sorted(result, key=lambda item: item[0])

def page_footnotes(page) -> dict[str, str]:
    chunks = []
    for block in page.get_text('blocks'):
        if 600 <= block[1] < 704 and block[4].strip():
            chunks.append(block[4])
    text = normalise(' '.join(chunks))
    starts = list(re.finditer(r'(?<!\S)(\d+)\s+', text))
    return {match.group(1): text[match.end(): starts[index + 1].start() if index + 1 < len(starts) else len(text)].strip()
            for index, match in enumerate(starts)}


In [ ]:
expected = {
    'Luke': [(1, verse) for verse in range(1, 36)],
    'John': [(1, verse) for verse in range(1, 14)],
}
records = {book: OrderedDict(
    ((chapter, verse), {'chapter': chapter, 'verse': verse, 'hebrew': '', 'english': '', 'notes': OrderedDict(), 'pages': set()})
    for chapter, verse in coverage
) for book, coverage in expected.items()}

# The source makes every verse number visible in the English column. Hebrew
# labels drive the first pass; English labels provide a reliable pairing audit.
current_hebrew = {'Luke': None, 'John': None}
current_english = {'Luke': None, 'John': None}
with fitz.open(PDF_PATH) as document:
    for page_number, page in enumerate(document, start=1):
        starts = []
        # Luke 1:27 begins at the top of PDF page 8, but its numeral is
        # absent from the Hebrew text layer; the adjacent English '(27)'
        # and the following visible Hebrew 28 anchor this source position.
        if page_number == 8:
            current_hebrew['Luke'] = (1, 27)
            starts.append((60, 'Luke', (1, 27)))
        for y, text, marker in content_lines(page, 'hebrew'):
            book = book_for_line(page_number, y)
            if book is None or 'פֶרֶק רִאש' in text:
                continue
            if marker and (1, marker) in records[book]:
                current_hebrew[book] = (1, marker)
                starts.append((y, book, current_hebrew[book]))
            key = current_hebrew[book]
            if key is not None:
                append(records[book][key], 'hebrew', text)
                records[book][key]['pages'].add(page_number)

        for y, text, marker in content_lines(page, 'english'):
            book = book_for_line(page_number, y)
            if book is None or text == 'Chapter 1':
                continue
            candidates = [item for item in starts if item[1] == book and item[0] <= y + 12]
            if candidates:
                current_english[book] = candidates[-1][2]
                text = ENGLISH_MARKER_RE.sub('', text, count=1)
            key = current_english[book]
            if key is not None:
                append(records[book][key], 'english', text)
                records[book][key]['pages'].add(page_number)

with fitz.open(PDF_PATH) as document:
    all_notes = {}
    for page in document:
        all_notes.update(page_footnotes(page))

# Audited targets for the small superscripts; notes 1–2 annotate the Luke title.
title_notes = OrderedDict((number, all_notes[number]) for number in ('1', '2'))
note_targets = {
    '3': ('Luke', 1), '4': ('Luke', 2), '5': ('Luke', 2), '6': ('Luke', 1),
    '7': ('Luke', 8), '8': ('Luke', 8), '9': ('Luke', 13), '10': ('Luke', 22),
    '11': ('Luke', 28), '12': ('Luke', 28), '13': ('Luke', 35), '14': ('Luke', 35), '15': ('Luke', 35),
    '16': ('John', 3), '17': ('John', 3), '18': ('John', 5), '19': ('John', 10),
    '20': ('John', 11), '21': ('John', 12), '22': ('John', 13),
}
for number, (book, verse) in note_targets.items():
    records[book][(1, verse)]['notes'][number] = all_notes[number]

for book, book_records in records.items():
    for record in book_records.values():
        record['hebrew'] = clean_hebrew(record['hebrew'])
        record['english'] = normalise(record['english'])
    missing = [key for key, record in book_records.items() if not record['hebrew'] or not record['english']]
    assert not missing, f'{book} missing text: {missing}'

assert {str(number) for number in range(1, 23)} <= set(all_notes)
print(f"Luke: {len(records['Luke'])} verses; John: {len(records['John'])} verses")
print(f'Verse-linked footnotes: {len(note_targets)}; title footnotes: {len(title_notes)}')


In [ ]:
def tag(name: str) -> str:
    return f'{{{OSIS_NS}}}{name}'

def make_header(parent, *, book: str, work_id: str, language: str, translation: bool) -> None:
    header = etree.SubElement(parent, tag('header'))
    work = etree.SubElement(header, tag('work'), osisWork=work_id)
    title = f"English Translation of {book} (Vatican Biblioteca Apostolica ebr. 530)" if translation else f"{book} (Vatican Biblioteca Apostolica ebr. 530)"
    etree.SubElement(work, tag('title')).text = title
    etree.SubElement(work, tag('identifier'), type='OSIS').text = work_id
    etree.SubElement(work, tag('scope')).text = 'LUK' if book == 'Luke' else 'JOH'
    etree.SubElement(work, tag('language')).text = language
    etree.SubElement(work, tag('type'), type='x-translation' if translation else 'x-manuscript').text = 'Translation' if translation else 'Manuscript'
    etree.SubElement(work, tag('identifier'), type='shelfmark').text = 'Biblioteca Apostolica Vaticana, ebr. 530, part 1, fragment 11, folios 1r–2v'
    etree.SubElement(work, tag('contributor'), role='trl' if translation else 'trc', **{'file-as': 'Gordon, Nehemia'}).text = 'Nehemia Gordon'
    etree.SubElement(work, tag('date'), event='eversion', type='ISO').text = '2018'
    etree.SubElement(work, tag('rights')).text = '© 2018 Nehemia Gordon. All rights reserved.'
    etree.SubElement(work, tag('description')).text = (
        f"Hebrew transcription and English translation of {book} from Biblioteca Apostolica Vaticana ebr. 530, part 1, fragment 11, folios 1r–2v, a manuscript held by the Vatican Library. Transcribed, translated, and annotated by Nehemia Gordon."
    )
    bible = etree.SubElement(header, tag('work'), osisWork='bible')
    etree.SubElement(bible, tag('identifier'), type='OSIS').text = 'bible'
    etree.SubElement(bible, tag('refSystem')).text = 'StandardV11N'
    etree.SubElement(bible, tag('language')).text = language

def build_osis(book: str, variant: str) -> etree.ElementTree:
    translation = variant == 'translation'
    commented = variant == 'hebrew_commented'
    prefix = 'LUK_Ebr530' if book == 'Luke' else 'JOH_Ebr530'
    suffix = {'hebrew': 'Hebrew', 'hebrew_commented': 'Hebrew_Commented', 'translation': 'Translation'}[variant]
    work_id = f'{prefix}_{suffix}'
    language = 'en' if translation else 'he'
    root = etree.Element(tag('osis'), nsmap={None: OSIS_NS, 'xsi': XSI_NS})
    root.set(f'{{{XSI_NS}}}schemaLocation', f'{OSIS_NS} http://www.bibletechnologies.net/osisCore.2.1.1.xsd')
    osis_text = etree.SubElement(root, tag('osisText'), osisIDWork=work_id, osisRefWork='bible')
    osis_text.set('{http://www.w3.org/XML/1998/namespace}lang', language)
    make_header(osis_text, book=book, work_id=work_id, language=language, translation=translation)
    abbreviation = 'Luke' if book == 'Luke' else 'John'
    book_el = etree.SubElement(osis_text, tag('div'), type='book', osisID=abbreviation)
    if book == 'Luke' and (commented or translation):
        for number, note_text in title_notes.items():
            etree.SubElement(book_el, tag('note'), type='footnote', n=number).text = note_text
    chapter_el = etree.SubElement(book_el, tag('chapter'), osisID=f'{abbreviation}.1')
    for record in records[book].values():
        verse_el = etree.SubElement(chapter_el, tag('verse'), osisID=f"{abbreviation}.{record['chapter']}.{record['verse']}", n=str(record['verse']))
        verse_el.text = record['english'] if translation else record['hebrew']
        if commented or translation:
            for number, note_text in record['notes'].items():
                note = etree.SubElement(verse_el, tag('note'), type='footnote', n=number)
                note.text = note_text
                note.tail = ' '
    return etree.ElementTree(root)

outputs = {}
for book, stem in (('Luke', 'LUK_Ebr530'), ('John', 'JOH_Ebr530')):
    for variant in ('hebrew', 'hebrew_commented', 'translation'):
        path = OUT_DIR / f'{stem}_{variant}.osis'
        build_osis(book, variant).write(path, encoding='UTF-8', xml_declaration=True, pretty_print=True)
        outputs[(book, variant)] = path
        print(f'Wrote {path}')


In [ ]:
schema = etree.XMLSchema(etree.parse(str(SCHEMA)))
ns = {'osis': OSIS_NS}
for (book, variant), path in outputs.items():
    tree = etree.parse(str(path))
    assert schema.validate(tree), schema.error_log
    abbreviation = 'Luke' if book == 'Luke' else 'John'
    ids = tree.xpath('//osis:verse/@osisID', namespaces=ns)
    expected_ids = [f'{abbreviation}.{chapter}.{verse}' for chapter, verse in expected[book]]
    assert ids == expected_ids
    notes = tree.xpath('//osis:note', namespaces=ns)
    expected_notes = 0 if variant == 'hebrew' else sum(1 for target_book, _ in note_targets.values() if target_book == book)
    if book == 'Luke' and variant != 'hebrew':
        expected_notes += len(title_notes)
    assert len(notes) == expected_notes, (path, len(notes), expected_notes)
    print(f'{path.name}: {len(ids)} verses, {len(notes)} notes, schema valid')

assert records['Luke'][(1, 1)]['hebrew'] and records['Luke'][(1, 1)]['english']
assert len(records['Luke'][(1, 35)]['pages']) >= 1
assert records['John'][(1, 1)]['hebrew'] and records['John'][(1, 13)]['english']
assert records['John'][(1, 3)]['notes']['16']
print('Luke 1:1:', records['Luke'][(1, 1)]['hebrew'][:80])
print('John 1:1:', records['John'][(1, 1)]['english'][:80])
print('Footnote 16:', records['John'][(1, 3)]['notes']['16'])
